# How many Likes to know you?

A simulated version of the study: a personality trait predicted from a
person's Likes, with accuracy rising with the number of Likes available. The
real paper is Youyou, Kosinski and Stillwell (2015).

In [1]:
import numpy as np
rng = np.random.default_rng(5)

n_people, n_pages = 3000, 400
page_signal = rng.normal(0, 1, n_pages)                       # each page carries a little information about the trait
trait = rng.normal(0, 1, n_people)
p_like = 1 / (1 + np.exp(-(0.22 * np.outer(trait, page_signal) - 1.6)))
likes = (rng.random((n_people, n_pages)) < p_like).astype(float)   # binary Likes matrix
print(likes.shape, "average Likes per person:", likes.sum(axis=1).mean().round(1))

(3000, 400) average Likes per person: 67.8


## Learn the trait from Likes (linear regression), evaluate on held-out people

In [2]:
train, test = slice(0, 2000), slice(2000, None)

def fit_predict(n_available):
    X = likes[:, :n_available]
    Xb = np.column_stack([X, np.ones(len(X))])
    w = np.linalg.lstsq(Xb[train], trait[train], rcond=None)[0]
    pred = Xb[test] @ w
    return np.corrcoef(pred, trait[test])[0, 1]

for n in [10, 30, 70, 150, 300, 400]:
    print(f"{n:>4} Likes available: r = {fit_predict(n):.2f}")

  10 Likes available: r = 0.20
  30 Likes available: r = 0.30
  70 Likes available: r = 0.49
 150 Likes available: r = 0.63
 300 Likes available: r = 0.77
 400 Likes available: r = 0.80


Accuracy rises with the logarithm of the number of Likes, which is the shape
of the curve in the paper, where 10 Likes matched a work colleague, 70 a
friend, 150 a family member and 300 a spouse (human judges averaged
$r \approx 0.49$).

## The privacy consequence

Each Like says almost nothing; the pattern says a lot. And a model fitted on
*other people* still works on you:

In [3]:
print(f"a model that never saw the test people: r = {fit_predict(400):.2f} on them")

a model that never saw the test people: r = 0.80 on them


**What we used:** supervised learning as a measurement instrument (session 10,
section 3) and its privacy consequences, inference and shadow profiles
(session 13, sections 10 and 11).